<div style="border-radius: 12px; overflow: hidden; margin-bottom: 20px; box-shadow: 0 6px 24px rgba(0,0,0,0.3);">

  <!-- Gold accent bar -->
  <div style="height: 5px; background: linear-gradient(90deg, #c9a227 0%, #f0d060 50%, #c9a227 100%);"></div>

  <!-- Main banner -->
  <div style="background: linear-gradient(135deg, #0d1f35 0%, #1e3a5f 60%, #0f2340 100%); padding: 32px 40px 28px;">
    <div style="display: flex; align-items: center; justify-content: space-between; flex-wrap: nowrap; gap: 24px;">
      <!-- Left: logo + title -->
      <div style="display: flex; align-items: center; gap: 24px; flex: 1 1 auto; min-width: 0;">
        <img src="https://scidx.sci.utah.edu/wp-content/uploads/2024/12/logo-sm.png" alt="SciDX Logo" width="155" />
        <div>
          <div style="color: #ffffff; font-size: 0.7em; font-weight: 700; letter-spacing: 3px; text-transform: uppercase; margin-bottom: 6px;">SciDX &nbsp;·&nbsp; National Data Platform</div>
          <h1 style="margin: 0; color: #ffffff; font-size: 2.1em; font-weight: 800; letter-spacing: -0.5px; line-height: 1.1;">Air Quality Workflow</h1>
          <div style="margin-top: 10px; color: #93c5fd; font-size: 0.85em;">Generate predictions, detect hotspots, and create streams with the <code style="background: rgba(255,255,255,0.12); padding: 2px 7px; border-radius: 4px; color: #fde68a;">forecast_service</code>.</div>
        </div>
      </div>
      <!-- Right: NDP logo on white pill -->
      <div style="background: white; border-radius: 8px; padding: 12px 20px; box-shadow: 0 2px 10px rgba(0,0,0,0.2); flex: 0 0 auto; width: 250px; max-width: 250px; box-sizing: border-box; display: flex; align-items: center; justify-content: center;">
        <img src="https://nationaldataplatform.org/National_Data_Platform_horiz_stacked.svg" alt="NDP Logo" width="210" style="display: block; width: 210px; max-width: 100%; height: auto;" />
      </div>
    </div>
  </div>

  <!-- "What this notebook covers" footer strip -->
  <div style="background: #2a4a6e; padding: 18px 40px;">
    <div style="color: #ffffff; font-size: 0.68em; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 10px;">What this notebook covers</div>
    <ol style="margin: 0; padding-left: 20px; color: #93c5fd; font-size: 0.85em; line-height: 2;">
      <li>Initializing the NDP Endpoint and streaming clients</li>
      <li>Forecast service discovery and health checks</li>
      <li>Prediction job submission and async status polling</li>
      <li>Prediction CSV loading and hotspot detection</li>
      <li>Sensor selection and Kafka stream creation</li>
      <li>Real-time message consumption and payload inspection</li>
    </ol>
  </div>

</div>

<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 1</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Initialize the Air Quality Workflow</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Install dependencies, load environment variables, and configure API connection parameters for the Forecast Service.</p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 1.0 — Import dependencies and load configuration</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Loads environment variables, normalizes the API URL, prepares authorization headers, and prints the active connection settings.</p>
</div>

In [1]:
import os
import json
import asyncio
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta, timezone
from pprint import pprint

from dotenv import load_dotenv
from IPython.display import display

load_dotenv(override=True)

API_URL = (os.environ.get("API_URL") or "").strip()
TOKEN = (os.environ.get("TOKEN") or os.environ.get("ACCESS_TOKEN") or "").strip()
SERVER = (os.environ.get("SERVER") or "local").strip()
SERVICE_BASE_URL = (os.environ.get("SERVICE_BASE_URL") or "").strip()
SERVICE_NAME = (os.environ.get("SERVICE_NAME") or "forecast_service").strip().replace("-", "_")

if API_URL and not API_URL.startswith(("http://", "https://")):
    API_URL = f"http://{API_URL}"

if not API_URL:
    raise ValueError("API_URL is not set")
if not TOKEN:
    raise ValueError("TOKEN is not set")

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}

print(f"API_URL: {API_URL}")
print(f"SERVER: {SERVER}")
print(f"SERVICE_NAME: {SERVICE_NAME}")

API_URL: https://dev.air-quality.ndp.utah.edu/ep
SERVER: local
SERVICE_NAME: forecast_service


<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 2</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Connect to NDP and Streaming Services</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Instantiate the NDP Endpoint client and streaming client used for catalog lookup, service calls, and Kafka stream creation.</p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 2.0 — Instantiate the clients</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Creates <code>APIClient</code> and <code>StreamingClient</code>, then confirms the streaming user context.</p>
</div>

In [2]:
from ndp_ep import APIClient
from scidx_streaming import StreamingClient

client = APIClient(base_url=API_URL, token=TOKEN)
streaming = StreamingClient(client)

print(f"user_id: {streaming.user_id}")

user_id: fc624925-ef09-447d-bf16-378066799275


<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 3</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Check Forecast Service Availability</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Search the catalog for the Forecast Service and verify that its redirected health endpoint is reachable before continuing.</p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 3.0 — Discover the service dataset</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Queries the selected server for the configured service name and previews matching catalog records.</p>
</div>

In [10]:
service_matches = client.search_datasets([SERVICE_NAME], server=SERVER) or []
print(f"matches: {len(service_matches)}")

if service_matches:
    service_df = pd.DataFrame([
        {
            "name": item.get("name"),
            "title": item.get("title"),
            "owner_org": item.get("owner_org"),
        }
        for item in service_matches
    ])
    display(service_df.head())

matches: 2


,name,title,owner_org
0,forecast_service,Forecast API Service,services
1,forecast_service_20260422_163758,Forecast API Service,services


<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 3.1 — Verify the service health endpoint</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Calls the Forecast Service health endpoint through NDP service redirection and prints the response payload.</p>
</div>

In [5]:
health_url = f"{API_URL}/services/redirect/{SERVICE_NAME}/health"
response = requests.get(health_url, headers=HEADERS, timeout=30)
print(f"GET {health_url} -> {response.status_code}")
response.raise_for_status()
print(json.dumps(response.json(), indent=2))

GET https://dev.air-quality.ndp.utah.edu/ep/services/redirect/forecast_service/health -> 200
{
  "ok": true
}


<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 4</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Generate Predictions</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Launch the Forecast Service workflow, poll the async job until it finishes, and capture the generated prediction dataset for downstream use.</p>
</div>

<div style="background: #fffbeb; border-left: 4px solid #f59e0b; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 8px 0 12px;">
  <p style="margin: 0; color: #78350f; font-size: 0.85em;">
    <strong>Long-running step</strong>: Prediction generation runs asynchronously and may take several minutes, depending on the requested dates and model inputs.
  </p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 4.0 — Configure the workflow run</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Set the region bounds, date range, Aurora forecast options, and overwrite behavior for the prediction job.</p>
</div>

In [ ]:
RUN_REGION = {
    "name": "utah",
    "lat_min": 37.0,
    "lat_max": 42.1,
    "lon_min": 245.9,
    "lon_max": 251.0,
}

RUN_DATES = {
    "start": "2025-01-16",
    "end": "2025-01-16",
}

RUN_OVERWRITE = False

run_payload = {
    "region": RUN_REGION,
    "dates": RUN_DATES,
    "overwrite": RUN_OVERWRITE,
}

print(json.dumps(run_payload, indent=2))


{
  "region": {
    "name": "utah",
    "lat_min": 37.0,
    "lat_max": 42.1,
    "lon_min": 245.9,
    "lon_max": 251.0
  },
  "dates": {
    "start": "2025-01-16",
    "end": "2025-01-16"
  },
  "aurora": {
    "times": [
      "00:00",
      "12:00"
    ],
    "lead_hours": 12,
    "region_tag": "global"
  },
  "overwrite": false
}


<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 4.1 — Submit the prediction job</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Posts to <code>run-model</code>, stores the returned <code>JOB_ID</code>, and prints it so the next cell can resume status checks independently.</p>
</div>

In [12]:
run_url = f"{API_URL}/services/redirect/{SERVICE_NAME}/run-model"
response = requests.post(run_url, headers=HEADERS, json=run_payload, timeout=60)
print(f"POST {run_url} -> {response.status_code}")
response.raise_for_status()

run_result = response.json()
JOB_ID = run_result.get("job_id")
if not JOB_ID:
    raise ValueError(f"No job_id returned: {run_result}")

print(json.dumps(run_result, indent=2))
print(f"job_id: {JOB_ID}")


POST https://dev.air-quality.ndp.utah.edu/ep/services/redirect/forecast_service/run-model -> 200
{
  "job_id": "95afc478-2385-458c-90eb-218f16478c70",
  "status": "queued",
  "time_tag": "0000-1200",
  "lead_hours": 12
}
job_id: 95afc478-2385-458c-90eb-218f16478c70


<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 4.2 — Poll job status and load the prediction output</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Uses the <code>JOB_ID</code> from the previous cell to query <code>status/{job_id}</code>, then loads the generated prediction CSV once the job finishes.</p>
</div>

In [ ]:
import time

if not JOB_ID:
    raise ValueError("JOB_ID is not set. Run the submit-job cell first.")

status_url = f"{API_URL}/services/redirect/{SERVICE_NAME}/status/{JOB_ID}"
POLL_SECONDS = 10
MAX_POLLS = 90
job_info = None
status = "queued"
poll_interrupted = False

try:
    for attempt in range(1, MAX_POLLS + 1):
        response = requests.get(status_url, headers=HEADERS, timeout=30)
        response.raise_for_status()
        job_info = response.json()
        status = job_info.get("status", "unknown")
        step = job_info.get("step", "")
        current_date = job_info.get("current_date")
        print(f"[{attempt:02d}] status={status} step={step} current_date={current_date}")
        if status in {"done", "failed"}:
            break
        time.sleep(POLL_SECONDS)
    else:
        raise TimeoutError(f"Prediction job did not finish after {MAX_POLLS * POLL_SECONDS} seconds")
except KeyboardInterrupt:
    poll_interrupted = True
    print("\nPolling stopped by user.")
    print(f"job_id: {JOB_ID}")
    print("Remote prediction job is still running. Use the job_id above to resume status checks later.")

if not poll_interrupted:
    if status == "failed":
        raise RuntimeError(job_info.get("error") or "run-model failed")

    result = job_info.get("result") or {}
    prediction_urls = result.get("prediction_urls") or []
    prediction_paths = result.get("predictions") or []
    ndp_result = result.get("ndp") or result.get("ndp_registration") or {}
    registered_items = ndp_result.get("registered") if isinstance(ndp_result, dict) else []

    PREDICTION_DATASET_NAME = None
    if registered_items:
        first_item = registered_items[0] or {}
        PREDICTION_DATASET_NAME = first_item.get("dataset_name")

    prediction_url = prediction_urls[0] if prediction_urls else None
    prediction_path = prediction_paths[0] if prediction_paths else None
    if not prediction_url and registered_items:
        prediction_url = (registered_items[0] or {}).get("url")

    print(f"prediction_dataset: {PREDICTION_DATASET_NAME}")
    print(f"prediction_url: {prediction_url}")
    print(f"prediction_path: {prediction_path}")

    if not prediction_url:
        raise ValueError("No prediction URL returned from the workflow run")


[01] status=done step=complete current_date=2025-01-16
prediction_dataset: prediction_utah_20260503t005728
prediction_url: https://dev.air-quality.ndp.utah.edu/files?url=prediction%3A2025-01-16_0000-1200_12h_utah.csv
prediction_path: data/processed/predictions/2025-01-16_0000-1200_12h_utah.csv


<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 5</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Load Prediction Dataset</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Locate the prediction dataset, extract its CSV resource URL, and load the records into a DataFrame for inspection.</p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 5.0 — Use workflow output or catalog lookup</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Uses the workflow prediction URL when available, otherwise falls back to catalog lookup by dataset name before previewing the CSV.</p>
</div>

In [5]:
DEFAULT_PREDICTION_DATASET_NAME = "prediction_utah_20260323t015428"

if "prediction_url" not in globals():
    prediction_url = None
if "prediction_df" not in globals():
    prediction_df = None
if "PREDICTION_DATASET_NAME" not in globals() or not PREDICTION_DATASET_NAME:
    PREDICTION_DATASET_NAME = DEFAULT_PREDICTION_DATASET_NAME

prediction_datasets = client.search_datasets([PREDICTION_DATASET_NAME], server=SERVER) or []
prediction_datasets = [
    item for item in prediction_datasets
    if item.get("name") == PREDICTION_DATASET_NAME
]
print(f"matches: {len(prediction_datasets)}")

if not prediction_url and prediction_datasets:
    prediction_resources = prediction_datasets[0].get("resources") or []
    if prediction_resources:
        prediction_url = prediction_resources[0].get("url")

if not prediction_url:
    raise ValueError("No prediction CSV URL found")

print(prediction_url)
if prediction_df is None or getattr(prediction_df, "empty", True):
    prediction_df = pd.read_csv(prediction_url)
display(prediction_df.head(10))


matches: 1
https://dev.air-quality.ndp.utah.edu/files?url=prediction%3A2025-01-16_0000-1200_12h_utah.csv


,timestamp,latitude,longitude,pm1_ugm3,pm2p5_ugm3,pm10_ugm3,co,no,no2,o3,so2
0,2025-01-16 01:00:00,42.0,246.0,5.139501,8.416309,12.769647,0.000790,0.000002,0.000002,0.006457,0.000002
1,2025-01-16 01:00:00,42.0,246.4,4.973207,8.134764,12.682975,0.000786,0.000002,0.000002,0.006455,0.000002
2,2025-01-16 01:00:00,42.0,246.8,4.733636,7.780323,11.810864,0.000774,0.000002,0.000002,0.006446,0.000002
3,2025-01-16 01:00:00,42.0,247.2,4.271605,7.442184,11.207210,0.000757,0.000002,0.000002,0.006467,0.000002
4,2025-01-16 01:00:00,42.0,247.6,4.124492,7.034207,10.778785,0.000753,0.000002,0.000002,0.006455,0.000002
5,2025-01-16 01:00:00,42.0,248.0,3.794223,6.358487,9.616916,0.000768,0.000002,0.000002,0.006468,0.000002
6,2025-01-16 01:00:00,42.0,248.4,3.685385,5.724228,7.698381,0.000769,0.000002,0.000002,0.006450,0.000002
7,2025-01-16 01:00:00,42.0,248.8,3.315233,5.285413,7.333610,0.000791,0.000002,0.000002,0.006481,0.000002
8,2025-01-16 01:00:00,42.0,249.2,2.838629,4.566443,6.258007,0.000816,0.000002,0.000002,0.006517,0.000002
9,2025-01-16 01:00:00,42.0,249.6,1.975215,3.626189,5.587967,0.000799,0.000002,0.000002,0.006510,0.000002


<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 6</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Detect Hotspots</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Send the prediction CSV to the Forecast Service hotspot endpoint and capture the returned dataset or preview rows.</p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 6.0 — Call the hotspot detection endpoint</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Posts threshold and persistence settings to <code>detect-hotspot</code> and records the returned hotspot CSV location.</p>
</div>

In [6]:
HOTSPOT_THRESHOLD = 10.0
PERSISTENCE_K = 6

hotspot_url = None
hotspots_csv = None
hotspot_dataset_name = None
hotspot_rows = []

hotspot_payload = {
    "prediction_file": prediction_url,
    "threshold": HOTSPOT_THRESHOLD,
    "persistence_k": PERSISTENCE_K,
    "timestep_hours": 1,
    "threshold_field": "pm2p5_ugm3",
}

detect_url = f"{API_URL}/services/redirect/{SERVICE_NAME}/detect-hotspot"
response = requests.post(detect_url, headers=HEADERS, json=hotspot_payload, timeout=120)
print(f"POST {detect_url} -> {response.status_code}")
response.raise_for_status()

hotspot_result = response.json()
hotspot_rows = hotspot_result.get("preview") or []
hotspots_csv = hotspot_result.get("hotspots_csv")

ndp_result = hotspot_result.get("ndp") or {}
registered_items = ndp_result.get("registered") if isinstance(ndp_result, dict) else []

if registered_items:
    first_item = registered_items[0] or {}
    hotspot_url = first_item.get("url")
    hotspot_dataset_name = first_item.get("dataset_name")

hotspot_input = hotspot_url or hotspots_csv
if not hotspot_input:
    raise ValueError("No hotspot CSV path or URL found")

print(f"hotspot_dataset: {hotspot_dataset_name}")
print(f"hotspot_input: {hotspot_input}")

POST https://dev.air-quality.ndp.utah.edu/ep/services/redirect/forecast_service/detect-hotspot -> 200
hotspot_dataset: hotspots_pm2p5_ugm3_20260428t190732
hotspot_input: https://dev.air-quality.ndp.utah.edu/files?url=hotspot%3A2025-01-16_0000-1200_12h_utah_k6_hotspots.csv


<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 6.1 — Display hotspot results</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Loads the registered hotspot CSV, a local CSV path, or returned preview rows into a DataFrame.</p>
</div>

In [10]:
if hotspot_url:
    hotspot_df = pd.read_csv(hotspot_url)
elif hotspots_csv and Path(hotspots_csv).exists():
    hotspot_df = pd.read_csv(hotspots_csv)
elif hotspot_rows:
    hotspot_df = pd.DataFrame(hotspot_rows)
else:
    raise ValueError("No hotspot rows available")

print("Hotspots (10 rows):")
display(hotspot_df.head(10))

Hotspots (10 rows):


,timestamp,lat,lon,value,variable,threshold_field,threshold_value,pm25
0,2025-01-16 06:00:00+00:00,38.4,248.0,10.188977,pm2p5_ugm3,pm2p5_ugm3,10.0,10.188977
1,2025-01-16 06:00:00+00:00,41.2,247.6,12.742494,pm2p5_ugm3,pm2p5_ugm3,10.0,12.742494
2,2025-01-16 06:00:00+00:00,41.2,246.8,13.701937,pm2p5_ugm3,pm2p5_ugm3,10.0,13.701937
3,2025-01-16 06:00:00+00:00,40.8,247.6,13.104024,pm2p5_ugm3,pm2p5_ugm3,10.0,13.104024
4,2025-01-16 06:00:00+00:00,40.8,247.2,12.915147,pm2p5_ugm3,pm2p5_ugm3,10.0,12.915147
5,2025-01-16 06:00:00+00:00,40.8,246.8,12.464286,pm2p5_ugm3,pm2p5_ugm3,10.0,12.464286
6,2025-01-16 06:00:00+00:00,40.4,248.0,13.130033,pm2p5_ugm3,pm2p5_ugm3,10.0,13.130033
7,2025-01-16 06:00:00+00:00,40.4,247.6,12.487503,pm2p5_ugm3,pm2p5_ugm3,10.0,12.487503
8,2025-01-16 06:00:00+00:00,40.4,247.2,11.175134,pm2p5_ugm3,pm2p5_ugm3,10.0,11.175134
9,2025-01-16 06:00:00+00:00,41.2,247.2,13.923840,pm2p5_ugm3,pm2p5_ugm3,10.0,13.923840


<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 7</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Select Sensors</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Use the hotspot output to select nearby sensors (Haversine nearest-k on the Forecast Service) and inspect the returned sensor–hotspot mapping plus <code>map_csv</code> when available.</p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 7.0 — Call the sensor selection endpoint</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Posts the hotspot file to <code>select-sensor</code> with <code>selection_mode: nearest_k</code> and <code>k: 1</code> (one nearest sensor per hotspot). The JSON <code>preview</code> lists mapping rows; <code>mapping_count</code> and <code>map_csv</code> reflect the full result after deploy.</p>
</div>

In [11]:
sensor_map_csv = None
sensor_df = None

sensor_payload = {
    "hotspots_file": hotspot_input,
    "selection_mode": "nearest_k",
    "k": 1,
}

select_url = f"{API_URL}/services/redirect/{SERVICE_NAME}/select-sensor"
response = requests.post(select_url, headers=HEADERS, json=sensor_payload, timeout=120)
print(f"POST {select_url} -> {response.status_code}")
response.raise_for_status()

sensor_result = response.json()
sensor_map_csv = sensor_result.get("map_csv")
sensor_rows = sensor_result.get("preview") or []

if not sensor_rows:
    raise ValueError("No sensor rows returned")

sensor_df = pd.DataFrame(sensor_rows)
print("total sensors:", sensor_df.shape[0])

sensor_df["distance_km"] = pd.to_numeric(sensor_df["distance_km"], errors="coerce")
lowest_10 = sensor_df.sort_values("distance_km", ascending=True).head(10).reset_index(drop=True)
print("Lowest distance (10 rows):")
display(lowest_10)

POST https://dev.air-quality.ndp.utah.edu/ep/services/redirect/forecast_service/select-sensor -> 200
total sensors: 67
Lowest distance (10 rows):


,timestamp,hotspot_lat,hotspot_lon,station_id,sensor_name,sensor_lat,sensor_lon,distance_km,threshold_field,threshold_value,event_value,pm25
0,2025-01-16 08:00:00+00:00,39.6,249.2,QP2,synoptic_push_qp2_s2365_aq,39.595800,-110.77000,2.612,pm2p5_ugm3,10.0,12.041231,12.041231
1,2025-01-16 20:00:00+00:00,39.6,249.2,QP2,synoptic_push_qp2_s2365_aq,39.595800,-110.77000,2.612,pm2p5_ugm3,10.0,12.045234,12.045234
2,2025-01-16 21:00:00+00:00,40.8,248.0,QUTTC,synoptic_push_quttc_s2365_aq,40.777100,-111.94500,5.284,pm2p5_ugm3,10.0,11.794483,11.794483
3,2025-01-16 09:00:00+00:00,40.8,248.0,QUTTC,synoptic_push_quttc_s2365_aq,40.777100,-111.94500,5.284,pm2p5_ugm3,10.0,11.835224,11.835224
4,2025-01-16 06:00:00+00:00,40.4,248.0,QH3,synoptic_push_qh3_s2365_aq,40.496410,-112.03631,11.152,pm2p5_ugm3,10.0,13.130033,13.130033
5,2025-01-16 10:00:00+00:00,40.4,248.4,QLN,synoptic_push_qln_s2365_aq,40.338800,-111.71330,11.766,pm2p5_ugm3,10.0,11.642755,11.642755
6,2025-01-16 23:00:00+00:00,40.4,248.4,QLN,synoptic_push_qln_s2365_aq,40.338800,-111.71330,11.766,pm2p5_ugm3,10.0,11.836043,11.836043
7,2025-01-16 07:00:00+00:00,41.6,247.2,LMS,synoptic_push_lms_s2365_aq,41.701000,-112.86181,12.349,pm2p5_ugm3,10.0,12.501699,12.501699
8,2025-01-16 19:00:00+00:00,41.6,247.2,LMS,synoptic_push_lms_s2365_aq,41.701000,-112.86181,12.349,pm2p5_ugm3,10.0,12.311638,12.311638
9,2025-01-16 10:00:00+00:00,40.8,248.4,UATKF,synoptic_push_uatkf_s2365_aq,40.810122,-111.76695,14.097,pm2p5_ugm3,10.0,12.140542,12.140542


<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 8</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Create Kafka Stream</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Resolve a streaming <strong>resource</strong> UUID for the chosen sensor (passed as <code>consumption_method_ids</code> in the client), then create a Kafka stream for live message delivery.</p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 8.0 — Choose a selected sensor</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Reads the first available sensor name from the sensor selection output and validates that a name was found.</p>
</div>

In [12]:
if sensor_df is None or sensor_df.empty or "sensor_name" not in sensor_df.columns:
    raise ValueError("Need non-empty sensor_df with 'sensor_name' from the select-sensor cell.")

_unique = sensor_df["sensor_name"].dropna().astype(str).drop_duplicates().sort_values()
print("Unique sensor_name values from sensor_df:")
for name in _unique:
    print(f"  - {name}")

Unique sensor_name values from sensor_df:
  - synoptic_push_gslm_s2365_aq
  - synoptic_push_lms_s2365_aq
  - synoptic_push_msi04_s2365_aq
  - synoptic_push_qbg_s2365_aq
  - synoptic_push_qed_s2365_aq
  - synoptic_push_qen_s2365_aq
  - synoptic_push_qh3_s2365_aq
  - synoptic_push_qhb_s2365_aq
  - synoptic_push_qln_s2365_aq
  - synoptic_push_qp2_s2365_aq
  - synoptic_push_qsf_s2365_aq
  - synoptic_push_quttc_s2365_aq
  - synoptic_push_uatkf_s2365_aq


In [13]:
selected_sensor_name = "synoptic_push_qln_s2365_aq"

print(f"\nsensor_name for stream creation: {selected_sensor_name}")


sensor_name for stream creation: synoptic_push_qln_s2365_aq


<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 8.1 — Resolve the sensor consumption method</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Searches streaming consumption methods for the selected sensor and lists every <code>resources[].id</code> UUID. The streaming API names these <code>consumption_method_ids</code>; they are resource ids, not a separate &quot;method&quot; object id.</p>
</div>

In [14]:
consumption_methods = streaming.search_consumption_methods(
    [selected_sensor_name, "sensors"],
    server=SERVER,
)

if not consumption_methods:
    consumption_methods = streaming.search_consumption_methods(
        [selected_sensor_name],
        server=SERVER,
    )

if not consumption_methods:
    raise ValueError(f"No consumption method found for {selected_sensor_name}")

resource_ids = [
    resource.get("id")
    for method in consumption_methods
    for resource in (method.get("resources") or [])
    if resource.get("id")
]

if not resource_ids:
    raise ValueError(f"No resource ids found for {selected_sensor_name}")

print("\nresource ids (API consumption_method_ids):")
for i, rid in enumerate(resource_ids, 1):
    print(f"{i}. {rid}")


resource ids (API consumption_method_ids):
1. 5a8efff7-e9b9-4465-811a-43fbda5beb69
2. 451b719c-4d99-45f5-912c-08ad81cfea4e


In [15]:
consumption_method_id = "5a8efff7-e9b9-4465-811a-43fbda5beb69"
print(f"\nresource id for stream creation (consumption_method_ids[0]): {consumption_method_id}")


resource id for stream creation (consumption_method_ids[0]): 5a8efff7-e9b9-4465-811a-43fbda5beb69


<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 8.2 — Create the stream topic</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Creates a Kafka stream from the chosen resource UUID (example hard-codes the WebSocket source id; align with the id you picked in Step&nbsp;8.1) and prints the returned topic.</p>
</div>

In [18]:
stream = await streaming.create_kafka_stream(
    consumption_method_ids=[consumption_method_id],
    filter_semantics=[],
    server=SERVER,
)

if isinstance(stream, dict) and stream.get("error"):
    raise RuntimeError(f"Kafka stream creation failed: {stream['error']}")

topic = getattr(stream, "data_stream_id", None) or getattr(stream, "topic", None)
if not topic:
    raise RuntimeError("No topic returned")

print(f"topic: {topic}")

topic: data_stream_fc624925-ef09-447d-bf16-378066799275_1


[websocket] CONNECTED url=wss://push.synopticdata.com/feed/e3636084e69244f5bd4d4d2721cec435/?stid=QLN&vars=PM_25_concentration,ozone_concentration,CO_concentration,NO2_concentration
[websocket] batch -> records=1 msg_types={'auth': 1}
[websocket] batch -> records=2 msg_types={'data': 1, 'metadata': 1}
[websocket] batch -> records=2 msg_types={'data': 1, 'metadata': 1}
[websocket] batch -> records=2 msg_types={'data': 1, 'metadata': 1}
[websocket] batch -> records=1 msg_types={'data': 1}


<div style="background: linear-gradient(135deg, #2c6496 0%, #1e4a75 100%); border-radius: 8px; padding: 18px 28px; margin: 24px 0 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.15);">
  <div style="color: #a0d8f0; font-size: 0.7em; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 4px;">Step 9</div>
  <h2 style="margin: 0; color: #ffffff; font-size: 1.4em; font-weight: 700;">Read Kafka Messages</h2>
  <p style="margin: 8px 0 0; color: #c5ddf5; font-size: 0.88em; line-height: 1.5;">Consume messages from the created topic and inspect unique payloads for validation and debugging.</p>
</div>

<div style="background: #fffbeb; border-left: 4px solid #f59e0b; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 8px 0 12px;">
  <p style="margin: 0; color: #78350f; font-size: 0.85em;">
    <strong>Long-running cell</strong>: This consumer waits until messages arrive, the configured message limit is reached, or the cell is interrupted.
  </p>
</div>

<div style="background: #f0f7ff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; padding: 12px 20px; margin: 12px 0 8px;">
  <h3 style="margin: 0; color: #1e3a5f; font-size: 1.05em; font-weight: 600;">Step 9.0 — Consume and inspect payloads</h3>
  <p style="margin: 4px 0 0; color: #4b5563; font-size: 0.85em;">Reads the streaming DataFrame, prints unique payload dictionaries, and stops the consumer in cleanup.</p>
</div>

In [19]:
MAX_WAIT_SECONDS = 60 * 60
MAX_MESSAGES = 5

topic = "data_stream_fc624925-ef09-447d-bf16-378066799275_1"
consumer = streaming.consume_kafka_messages(topic)
print(f"topic: {topic}")


def payload_items(value):
    if isinstance(value, dict):
        return [value]
    if isinstance(value, list):
        return [item for item in value if isinstance(item, dict)]
    return []


async def read_messages():
    deadline = datetime.now(timezone.utc) + timedelta(seconds=MAX_WAIT_SECONDS)
    seen = set()
    stopped_by_user = False

    try:
        while datetime.now(timezone.utc) < deadline and len(seen) < MAX_MESSAGES:
            df = consumer.dataframe
            if df is not None and not df.empty and "payload" in df.columns:
                for value in df["payload"].dropna():
                    for message in payload_items(value):
                        key = json.dumps(message, sort_keys=True, default=str)
                        if key in seen:
                            continue
                        seen.add(key)
                        pprint(message, width=120, sort_dicts=False)
                        if len(seen) >= MAX_MESSAGES:
                            break
                    if len(seen) >= MAX_MESSAGES:
                        break
            await asyncio.sleep(2)
    except (asyncio.CancelledError, KeyboardInterrupt):
        stopped_by_user = True
    finally:
        consumer.stop()

    if stopped_by_user:
        print("stopped_by_user: True")
    print(f"messages_read: {len(seen)}")


await read_messages()

topic: data_stream_fc624925-ef09-447d-bf16-378066799275_1
{'type': 'auth',
 'session': '44cb4c6b-e166-43d1-b7a9-a4c5bbd8902a',
 'messages': ['Starting new session.'],
 'code': 'success'}
{'type': 'data',
 'data': [{'stid': 'QLN',
           'sensor': 'PM_25_concentration',
           'set': 1,
           'date': '2026-04-28 19:00:00',
           'value': '5.3',
           'qc': [],
           'match': False}]}
{'type': 'metadata',
 'units': [{'sensor': 'PM_25_concentration', 'unit': ''}],
 'stations': [{'stid': 'QLN',
               'latitude': '40.3388',
               'longitude': '-111.7133',
               'elevation': 4738.0,
               'network': 9,
               'name': 'Lindon'}]}
{'type': 'data',
 'data': [{'stid': 'QLN',
           'sensor': 'ozone_concentration',
           'set': 1,
           'date': '2026-04-28 19:00:00',
           'value': '51.0',
           'qc': [],
           'match': False}]}
{'type': 'metadata', 'units': [{'sensor': 'ozone_concentration', 'uni